In [4]:
import time
from pathlib import Path

import cv2
import numpy as np
import requests

API_URL = "http://localhost:8000/predict"

IMAGE_PATH = Path(
    "../data/DevanagariHandwrittenCharacterDataset/Test/character_1_ka/1339.png"
)

NUM_WARMUP = 20
NUM_RUNS = 200

In [5]:
image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)

if image is None:
    raise FileNotFoundError(IMAGE_PATH)

_, buffer = cv2.imencode(".png", image)
image_bytes = buffer.tobytes()

In [6]:
print("Running warmup...")

for _ in range(NUM_WARMUP):
    response = requests.post(
        API_URL,
        files={"file": ("image.png", image_bytes, "image/png")},
    )
    response.raise_for_status()

print("Warmup complete.\n")

Running warmup...
Warmup complete.



In [7]:
latencies = []

print(f"Running {NUM_RUNS} requests...")

for _ in range(NUM_RUNS):

    start = time.perf_counter()

    response = requests.post(
        API_URL,
        files={"file": ("image.png", image_bytes, "image/png")},
    )

    end = time.perf_counter()

    response.raise_for_status()

    latencies.append((end - start) * 1000)

Running 200 requests...


In [8]:
latencies = np.array(latencies)

print("\nAPI Benchmark")
print("-----------------------------")
print(f"Mean latency : {latencies.mean():.3f} ms")
print(f"Median       : {np.median(latencies):.3f} ms")
print(f"P95 latency  : {np.percentile(latencies,95):.3f} ms")
print(f"P99 latency  : {np.percentile(latencies,99):.3f} ms")
print(f"Min latency  : {latencies.min():.3f} ms")
print(f"Max latency  : {latencies.max():.3f} ms")
print(f"Std Dev      : {latencies.std():.3f} ms")
print(f"Throughput   : {1000/latencies.mean():.2f} requests/sec")


API Benchmark
-----------------------------
Mean latency : 10.744 ms
Median       : 8.438 ms
P95 latency  : 23.728 ms
P99 latency  : 32.584 ms
Min latency  : 6.176 ms
Max latency  : 68.360 ms
Std Dev      : 7.355 ms
Throughput   : 93.07 requests/sec
